<div style="background: linear-gradient(135deg, #0A1628, #0F2033); padding: 30px 35px; border-radius: 12px; border-left: 6px solid #0891B2;">
<h1 style="color: white; margin: 0; font-size: 32px;">🧬 AI-Powered Enzyme Engineering</h1>
<h2 style="color: #06D6A0; margin: 8px 0 0 0; font-weight: 400; font-size: 20px;">ESM-2 Mutation Scanning & Closed-Loop DBTL Workshop</h2>
<p style="color: #94A3B8; margin: 12px 0 0 0;">Bioinformatics & Computational Biology</p>
</div>

<div style="background: #F0F9FF; padding: 20px 25px; border-radius: 10px; border-left: 5px solid #0891B2; margin: 15px 0;">

**This notebook demonstrates the complete Design–Build–Test–Learn (DBTL) cycle for AI-guided enzyme engineering using AtHMT as a case study.**

The target enzyme **AtHMT** (halide methyltransferase from *Arabidopsis thaliana*) and its sequence are automatically fetched from the [Zhao-Group/closed-loop](https://github.com/Zhao-Group/closed-loop) GitHub repository — **just run each cell in order, no pasting required.**

As described in Singh et al. (2025) *Nature Communications*, **each DBTL round adds one additional mutation** to the best variants from the previous round:

| Round | Mutation Order | Design Method | Variants Screened |
|:---:|:---|:---|:---:|
| **1** | Single mutants | ESM-2 + EVmutation (zero-shot) | ~180 |
| **2** | **Double** mutants | Low-N Ridge (trained on Round 1) | ~90 |
| **3** | **Triple** mutants | Retrained Ridge (Rounds 1+2) | ~90 |
| **4** | **Quadruple** mutants | Retrained Ridge (Rounds 1+2+3) | ~90 |

> 📖 Singh et al. [Nature Comms 2025](https://doi.org/10.1038/s41467-025-61209-y) · Hsu et al. [Nature Biotech 2021](https://doi.org/10.1038/s41587-021-01146-5) · Lin et al. [Science 2023 (ESM-2)](https://doi.org/10.1126/science.ade2574)
</div>

<div style="background: white; padding: 20px 25px; border-radius: 10px; border: 2px solid #E8EFF5;">
<h3 style="color: #0A1628; border-bottom: 3px solid #0891B2; padding-bottom: 8px;">🔄 The DBTL Cycle — Accumulating Mutations Round by Round</h3>

The key insight from the paper: **each round adds one new mutation on top of the best variant from the previous round**. This means Round 2 creates double mutants, Round 3 triples, and Round 4 quadruples.

```
  Round 1 (zero-shot)         Round 2 (supervised)      Round 3                 Round 4
  ─────────────────          ──────────────────         ──────────────           ──────────────
  Single mutants:            Double mutants:            Triple mutants:          Quadruple:
    V140T  ──────────┐         V140T/S99T ────────┐       V140T/L101I/E206D ┐     V140T/L101I/
    L101I             ├─────▶  V140T/L101I         ├───▶  V140T/L101I/T213S  ├──▶ E206D/S63N
    S99T              │        V140T/E206D         │      ...                │    (16× activity)
    E206D  ───────────┘        ...                 ┘      ...                ┘
```

**Real results:** AtHMT best variant (**V140T/L101I/E206D/S63N**) = 16× improved ethyltransferase activity. YmPhytase best variant (**V141M/K226G/I15V/Q362R**) = 26× improved activity at neutral pH.

</div>

---
<div style="background: linear-gradient(135deg, #0A1628, #0F2033); padding: 20px 25px; border-radius: 10px; border-left: 6px solid #0891B2; margin: 20px 0;">
<h2 style="color: white; margin: 0;">🎯 PHASE 1 — DESIGN (Round 1: Single Mutants)</h2>
<p style="color: #06D6A0; margin: 5px 0 0 0;">Zero-shot scoring of all possible single amino acid substitutions with ESM-2</p>
</div>

In Round 1, we have no experimental data. We use **ESM-2** to score every possible single mutation across selected positions. ESM-2 masks one position at a time and predicts which amino acids are most compatible with the surrounding sequence context.

**Δscore = log P(mutant) − log P(wildtype)** — positive means ESM-2 favors the substitution.


### Step 1.1 — Setup: Install Packages & Download AtHMT Data

In [ ]:
#@title 📦 Install & download data
!pip -q install fair-esm biopython pandas tqdm scikit-learn scipy matplotlib

# Clone the Zhao-Group repository to get AtHMT.fasta
!git clone --depth 1 https://github.com/Zhao-Group/closed-loop.git 2>/dev/null || echo "Already cloned"

import torch, esm, re, math, os, itertools
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

print(f"✅ torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
else:
    print("   💡 Tip: Runtime → Change runtime type → GPU for faster scoring")

assert os.path.exists("closed-loop/AtHMT.fasta"), "❌ AtHMT.fasta not found"
print("✅ AtHMT data ready")

### Step 1.2 — Load ESM-2

We load the 650M-parameter ESM-2 model. It takes a protein sequence, masks one position, and predicts log-probabilities for all 20 amino acids at that position.


In [ ]:
#@title 🧠 Load ESM-2 model
MODEL_NAME = "esm2_t33_650M_UR50D"  #@param ["esm2_t12_35M_UR50D","esm2_t33_650M_UR50D"]
device = "cuda" if torch.cuda.is_available() else "cpu"

if MODEL_NAME == "esm2_t12_35M_UR50D":
    model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
else:
    model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()

model = model.to(device).eval()
batch_converter = alphabet.get_batch_converter()
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

print(f"✅ {MODEL_NAME} loaded on {device}")

### Step 1.3 — Load the AtHMT Wild-Type Sequence

In [ ]:
#@title 🧬 Load AtHMT sequence from FASTA
with open("closed-loop/AtHMT.fasta") as f:
    lines = [l.strip() for l in f if l.strip()]

header = lines[0].lstrip(">")
WT_SEQ = "".join(lines[1:]).upper().replace(" ", "")

assert all(c in set(AMINO_ACIDS) for c in WT_SEQ), "Non-standard amino acids found!"
print(f"✅ AtHMT loaded: {len(WT_SEQ)} amino acids")
print(f"   Sequence: {WT_SEQ[:40]}...{WT_SEQ[-20:]}")

### Step 1.4 — Full-Protein Saturation Scan with ESM-2

We run a **complete saturation scan** — scoring all 19 possible amino acid substitutions at every position in the protein. ESM-2 uses the masked marginal approach: mask one position → predict log-probabilities for all 20 amino acids → compute ΔlogP = log P(mutant) − log P(wildtype).

Results are cached per position, so scanning the full protein requires only **{len(WT_SEQ)} forward passes** (one per position), not 19 × {len(WT_SEQ)}.


In [ ]:
#@title 🔬 Full-protein saturation scan with ESM-2

# ── Generate ALL possible single mutations across the entire protein ──
all_muts = []
for pos in range(1, len(WT_SEQ) + 1):
    wt = WT_SEQ[pos - 1]
    for mut_aa in AMINO_ACIDS:
        if mut_aa != wt:
            all_muts.append({"mutation": f"{wt}{pos}{mut_aa}", "wt": wt, "pos": pos, "mut": mut_aa})

print(f"Generated {len(all_muts)} possible single mutations ({len(WT_SEQ)} positions × 19 substitutions)")

# ── Scoring functions ──
@torch.no_grad()
def masked_logprobs_at_position(seq, pos1):
    """Log-probs over 20 AAs at position pos1 (1-based) via masked forward pass."""
    data = [("protein", seq)]
    _, _, toks = batch_converter(data)
    toks = toks.to(device)
    toks_masked = toks.clone()
    toks_masked[0, pos1] = alphabet.mask_idx
    out = model(toks_masked, repr_layers=[], return_contacts=False)
    logits = out["logits"][0, pos1]
    log_probs = torch.log_softmax(logits, dim=-1)
    return {aa: float(log_probs[alphabet.get_idx(aa)].cpu()) for aa in AMINO_ACIDS}

def score_mutations(seq, muts):
    """Score mutations with per-position caching."""
    cache = {}
    rows = []
    for m in tqdm(muts, desc="🔄 Scoring"):
        pos, wt, mut = m["pos"], m["wt"], m["mut"]
        if seq[pos-1] != wt:
            rows.append({**m, "status": "mismatch", "delta_logp": math.nan}); continue
        if pos not in cache:
            cache[pos] = masked_logprobs_at_position(seq, pos)
        lp = cache[pos]
        rows.append({**m, "status": "ok", "delta_logp": lp[mut] - lp[wt],
                     "logp_wt": lp[wt], "logp_mut": lp[mut]})
    return pd.DataFrame(rows).sort_values("delta_logp", ascending=False, na_position="last").reset_index(drop=True)

# ── Run full-protein scan ──
print(f"\nScoring {len(all_muts)} mutations on AtHMT ({len(WT_SEQ)} aa)...")
print(f"This requires {len(WT_SEQ)} ESM-2 forward passes (one per position)\n")

df_all_scores = score_mutations(WT_SEQ, all_muts)
df_all_scores = df_all_scores[df_all_scores["status"]=="ok"].reset_index(drop=True)

print(f"\n✅ Full scan complete: {len(df_all_scores)} mutations scored")
print(f"   Favorable (ΔlogP > 0): {(df_all_scores['delta_logp'] > 0).sum()}")
print(f"   Unfavorable (ΔlogP < 0): {(df_all_scores['delta_logp'] < 0).sum()}")

# Save full scan
df_all_scores.to_csv("AtHMT_full_saturation_scores.csv", index=False)
print(f"💾 Saved: AtHMT_full_saturation_scores.csv")

### Step 1.5 — Select Top 180 Mutations for Round 1

<div style="background: #F0F9FF; padding: 14px 18px; border-radius: 8px; border-left: 4px solid #0891B2;">

**Following the paper's design:** In Singh et al. (2025), the Round 1 library consisted of **180 single mutants** — 90 from ESM-2 and 90 from EVmutation. Since this tutorial demonstrates ESM-2 only, we select the **top 180 mutations ranked by ΔlogP** from the full saturation scan.

These 180 mutations become our Round 1 library — the starting point for the entire DBTL cycle.

</div>


In [ ]:
#@title 📊 Select top 180 & visualize heatmap
import matplotlib.pyplot as plt

# ══════════════════════════════════════════════════════════════
# Select the top 180 single mutations by ESM-2 ΔlogP score
# This is our Round 1 library (matching the paper's 180 variants)
# ══════════════════════════════════════════════════════════════
N_ROUND1 = 180  #@param {type:"integer"}
df_r1 = df_all_scores.head(N_ROUND1).copy().reset_index(drop=True)

print(f"✅ Selected top {N_ROUND1} mutations for Round 1")
print(f"   Score range: [{df_r1['delta_logp'].min():.3f}, {df_r1['delta_logp'].max():.3f}]")
print(f"   Unique positions hit: {df_r1['pos'].nunique()}")

# ── Heatmap of the top 180 ──
pos_list = sorted(df_r1["pos"].unique())
mat = np.full((len(pos_list), len(AMINO_ACIDS)), np.nan)
p2i = {p: i for i, p in enumerate(pos_list)}
a2j = {a: j for j, a in enumerate(AMINO_ACIDS)}
for _, r in df_r1.iterrows():
    mat[p2i[r["pos"]], a2j[r["mut"]]] = r["delta_logp"]

fig, ax = plt.subplots(figsize=(14, max(4, 0.3 * len(pos_list))))
vmax = max(abs(np.nanmin(mat)), abs(np.nanmax(mat)), 1)
im = ax.imshow(mat, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.set_yticks(range(len(pos_list))); ax.set_yticklabels(pos_list, fontsize=7)
ax.set_xticks(range(len(AMINO_ACIDS))); ax.set_xticklabels(AMINO_ACIDS, fontsize=9)
ax.set_xlabel("Mutant Amino Acid"); ax.set_ylabel("Position")
ax.set_title(f"Round 1 Library — Top {N_ROUND1} Mutations by ESM-2 ΔlogP", fontweight="bold", fontsize=14)
plt.colorbar(im, shrink=0.8, label="ΔlogP (blue=disruptive, red=favorable)")
plt.tight_layout(); plt.show()

print(f"\n🏆 Top 10 mutations:")
df_r1.head(10)[["mutation", "delta_logp"]]

In [ ]:
#@title 🫧 Bubble plot for Round 1 top mutations

import matplotlib.pyplot as plt
import numpy as np

# Input expected from previous block:
# df_r1       -> top mutations dataframe
# AMINO_ACIDS -> list of 20 amino acids

# Get sorted unique positions from the selected Round 1 mutations
pos_list = sorted(df_r1["pos"].unique())

# Mapping dictionaries
p2i = {p: i for i, p in enumerate(pos_list)}
a2i = {aa: j for j, aa in enumerate(AMINO_ACIDS)}

# Coordinates and values
x = [a2i[aa] for aa in df_r1["mut"]]
y = [p2i[p] for p in df_r1["pos"]]
scores = df_r1["delta_logp"].values

# Bubble sizes
sizes = 60 + 45 * np.abs(scores)

fig, ax = plt.subplots(figsize=(14, max(5, 0.30 * len(pos_list))))

sc = ax.scatter(
    x, y,
    c=scores,
    s=sizes,
    cmap="Reds",          # all selected mutations are positive, so Reds works well
    alpha=0.8,
    edgecolors="black",
    linewidths=0.4
)

# Axis formatting
ax.set_xticks(range(len(AMINO_ACIDS)))
ax.set_xticklabels(AMINO_ACIDS, fontsize=9)
ax.set_yticks(range(len(pos_list)))
ax.set_yticklabels(pos_list, fontsize=7)

ax.set_xlabel("Mutant Amino Acid")
ax.set_ylabel("Position")
ax.set_title(f"Round 1 Library — Top {len(df_r1)} Mutations (Bubble Plot)", fontweight="bold")

# Put smaller position numbers at top like the heatmap
ax.invert_yaxis()

# Light grid for readability
ax.grid(True, linestyle="--", alpha=0.25)

# Colorbar
cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
cbar.set_label("ΔlogP")

plt.tight_layout()
plt.show()

---
<div style="background: linear-gradient(135deg, #0A1628, #0F2033); padding: 20px 25px; border-radius: 10px; border-left: 6px solid #F4A261; margin: 20px 0;">
<h2 style="color: white; margin: 0;">⚙️ Utility Functions for Multi-Round DBTL</h2>
<p style="color: #F4A261; margin: 5px 0 0 0;">Helper functions used across all rounds: mutation application, one-hot encoding, training, and combinatorial generation</p>
</div>

These functions are used in every round of the DBTL cycle. They handle:
- **Applying mutations** to the wild-type sequence
- **One-hot encoding** protein sequences for Ridge regression
- **Training** the Low-N supervised model
- **Generating combinatorial candidates** (best singles + new mutation → doubles, etc.)


In [ ]:
#@title ⚙️ Define utility functions for all DBTL rounds
from sklearn.linear_model import RidgeCV
from scipy.stats import spearmanr

AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")

def apply_mutations(wt_seq, mut_str):
    """Apply one or more mutations to WT sequence.
    mut_str can be 'A10G' (single) or 'A10G:V140T' (multi, colon-separated)."""
    out = list(wt_seq)
    for m in [x for x in re.split(r"[:;,\s]+", mut_str.strip()) if x]:
        wt_aa, mut_aa, pos = m[0], m[-1], int(m[1:-1])
        assert out[pos-1] == wt_aa, f"{m}: expected {wt_aa} at pos {pos}, got {out[pos-1]}"
        out[pos-1] = mut_aa
    return "".join(out)

def one_hot_encode(seq):
    """Encode protein sequence as flat binary vector (20 × L features)."""
    X = np.zeros((len(seq), 20), dtype=np.float32)
    for i, aa in enumerate(seq):
        X[i, AA_LIST.index(aa)] = 1.0
    return X.reshape(-1)

def train_ridge(df, wt_seq):
    """Train RidgeCV on a DataFrame with 'mutations' and 'fitness' columns."""
    df = df.copy()
    df["sequence"] = df["mutations"].apply(lambda m: apply_mutations(wt_seq, m))
    X = np.vstack(df["sequence"].apply(one_hot_encode).values)
    y = np.log(df["fitness"].values + 1e-6)
    mdl = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0])
    mdl.fit(X, y)
    return mdl

def predict_fitness(mdl, candidates, wt_seq):
    """Predict fitness for a list of mutation strings."""
    seqs = [apply_mutations(wt_seq, m) for m in candidates]
    X = np.vstack([one_hot_encode(s) for s in seqs])
    return np.exp(mdl.predict(X))

def generate_combinatorial_candidates(base_mutations_list, single_mutations, max_candidates=200):
    """Generate higher-order mutants by adding one new single mutation to each base.

    Args:
        base_mutations_list: list of mutation strings (e.g., ["V140T", "V140T:L101I"])
        single_mutations: list of single mutation strings to add
        max_candidates: cap on total candidates

    Returns:
        list of new mutation strings (e.g., ["V140T:E206D", "V140T:L101I:S63N", ...])
    """
    candidates = set()
    for base in base_mutations_list:
        base_positions = set()
        for m in base.split(":"):
            pos = int(m[1:-1])
            base_positions.add(pos)
        for single in single_mutations:
            pos = int(single[1:-1])
            if pos not in base_positions:  # don't mutate same position twice
                new = f"{base}:{single}"
                candidates.add(new)
    # Return capped list
    return list(candidates)[:max_candidates]

print("✅ Utility functions defined")

---
<div style="background: linear-gradient(135deg, #0A1628, #0F2033); padding: 20px 25px; border-radius: 10px; border-left: 6px solid #06D6A0; margin: 20px 0;">
<h2 style="color: white; margin: 0;">🔨 ROUND 1 — BUILD & TEST (180 Single Mutants)</h2>
<p style="color: #06D6A0; margin: 5px 0 0 0;">All 180 selected mutations are constructed and tested experimentally (simulated here)</p>
</div>

In the real paper, **all 180 single mutants** were constructed and screened in two 96-well plates. 59.6% performed above wild-type. The best single mutant was **V140T** (2.1× improvement).

Here we simulate fitness values for all 180 mutants — just as the paper tested all of them, not a subset.


In [ ]:
#@title 🧪 Round 1: Simulate assay for all 180 single mutants
np.random.seed(42)

# ══════════════════════════════════════════════════════════════
# In the paper, ALL 180 Round 1 mutants were constructed and screened
# (two 96-well plates). We simulate fitness for all of them.
# ══════════════════════════════════════════════════════════════
df_tested_r1 = df_r1.copy()

# Simulated fitness (ESM-2 score as loose proxy + noise)
noise = np.random.normal(0, 0.3, size=len(df_tested_r1))
df_tested_r1["fitness"] = (0.4 * df_tested_r1["delta_logp"].values + 0.5 + noise).clip(min=0.01)
df_tested_r1["mutations"] = df_tested_r1["mutation"]  # standardize column name
df_tested_r1["round"] = 1
df_tested_r1["order"] = "single"

# Accumulate all tested variants across rounds
all_tested = df_tested_r1[["mutations", "fitness", "round", "order"]].copy()

print(f"✅ Round 1: Tested all {len(df_tested_r1)} single mutants")
print(f"   Above WT baseline (fitness > 0.5): {(df_tested_r1['fitness'] > 0.5).sum()} ({100*(df_tested_r1['fitness'] > 0.5).mean():.1f}%)")
print(f"   Best fitness: {df_tested_r1['fitness'].max():.3f}")
print(f"\n🏆 Top 10 variants:")
df_tested_r1.nlargest(10, "fitness")[["mutations", "delta_logp", "fitness"]]

---
<div style="background: linear-gradient(135deg, #0A1628, #0F2033); padding: 20px 25px; border-radius: 10px; border-left: 6px solid #0891B2; margin: 20px 0;">
<h2 style="color: white; margin: 0;">🎯 ROUND 2 — DESIGN & TEST (Double Mutants)</h2>
<p style="color: #0891B2; margin: 5px 0 0 0;">Train on Round 1 data → predict double mutants → simulate assay</p>
</div>

**This is the critical step where the paper's approach differs from naive re-ranking:**

1. **LEARN:** Train Ridge regression on all Round 1 assay data
2. **DESIGN:** Take the top N single mutants as "bases", combine each with every other promising single mutation → generate **double mutant candidates**
3. **PREDICT:** Score all candidates with the trained model, select top 90
4. **TEST:** Screen the top-ranked double mutants

In the paper, the best Round 2 variant was **S99T/V140T** (3× improvement over WT).


In [ ]:
#@title 🧠 Round 2: Train → Generate doubles → Predict → Test

# ── LEARN: Train on Round 1 data ──
model_r2 = train_ridge(all_tested, WT_SEQ)
print(f"✅ Ridge trained on {len(all_tested)} variants (Round 1)")
print(f"   α = {model_r2.alpha_}")

# ── DESIGN: Generate double mutant candidates ──
# Take top K single mutants as "bases" (the best singles to build upon)
TOP_BASES = 10  #@param {type:"integer"}
best_singles = df_tested_r1.nlargest(TOP_BASES, "fitness")["mutations"].tolist()

# Take top M single mutations as "additions" (the pool to combine with)
TOP_ADDITIONS = 30  #@param {type:"integer"}
addition_pool = df_r1.head(TOP_ADDITIONS)["mutation"].tolist()

# Combine: each base + each addition (where positions don't overlap)
double_candidates = generate_combinatorial_candidates(best_singles, addition_pool, max_candidates=500)
print(f"\n🔬 Generated {len(double_candidates)} double mutant candidates")
print(f"   (from {TOP_BASES} bases × {TOP_ADDITIONS} additions, filtering overlapping positions)")

# ── PREDICT: Rank doubles with the trained model ──
pred_fitness_r2 = predict_fitness(model_r2, double_candidates, WT_SEQ)
df_doubles = pd.DataFrame({"mutations": double_candidates, "predicted_fitness": pred_fitness_r2})
df_doubles = df_doubles.sort_values("predicted_fitness", ascending=False).reset_index(drop=True)

# ── TEST: Simulate assay for top 90 doubles ──
N_R2 = 90  #@param {type:"integer"}
df_tested_r2 = df_doubles.head(N_R2).copy()
noise = np.random.normal(0, 0.25, size=len(df_tested_r2))
df_tested_r2["fitness"] = (0.3 * df_tested_r2["predicted_fitness"].values + 0.6 + noise).clip(min=0.01)
df_tested_r2["round"] = 2
df_tested_r2["order"] = "double"

all_tested = pd.concat([all_tested, df_tested_r2[["mutations", "fitness", "round", "order"]]], ignore_index=True)

print(f"\n✅ Round 2: Tested {N_R2} double mutants")
print(f"   Best fitness: {df_tested_r2['fitness'].max():.3f}")
print(f"   Top 5 doubles:")
df_tested_r2.nlargest(5, "fitness")[["mutations", "predicted_fitness", "fitness"]]

---
<div style="background: linear-gradient(135deg, #0A1628, #0F2033); padding: 20px 25px; border-radius: 10px; border-left: 6px solid #F4A261; margin: 20px 0;">
<h2 style="color: white; margin: 0;">🧠 ROUND 3 — DESIGN & TEST (Triple Mutants)</h2>
<p style="color: #F4A261; margin: 5px 0 0 0;">Retrain on Rounds 1+2 → generate triples → predict → simulate assay</p>
</div>

Now we retrain the Ridge model on **all accumulated data** (Rounds 1 + 2) and predict **triple mutants** by adding one more mutation to the best doubles.

In the paper, the ML model's triple mutant predictions significantly outperformed human-intuited triples — **82% of model-predicted triples beat the best single mutant** V140T, vs. only 11% of manually designed triples.


In [ ]:
#@title 🧠 Round 3: Train → Generate triples → Predict → Test

# ── LEARN: Retrain on Rounds 1 + 2 ──
model_r3 = train_ridge(all_tested, WT_SEQ)
print(f"✅ Ridge retrained on {len(all_tested)} variants (Rounds 1+2)")
print(f"   α = {model_r3.alpha_}")

# ── DESIGN: Generate triple mutant candidates ──
TOP_DOUBLE_BASES = 8  #@param {type:"integer"}
best_doubles = df_tested_r2.nlargest(TOP_DOUBLE_BASES, "fitness")["mutations"].tolist()

triple_candidates = generate_combinatorial_candidates(best_doubles, addition_pool, max_candidates=500)
print(f"\n🔬 Generated {len(triple_candidates)} triple mutant candidates")

# ── PREDICT ──
pred_fitness_r3 = predict_fitness(model_r3, triple_candidates, WT_SEQ)
df_triples = pd.DataFrame({"mutations": triple_candidates, "predicted_fitness": pred_fitness_r3})
df_triples = df_triples.sort_values("predicted_fitness", ascending=False).reset_index(drop=True)

# ── TEST ──
N_R3 = 90  #@param {type:"integer"}
df_tested_r3 = df_triples.head(N_R3).copy()
noise = np.random.normal(0, 0.2, size=len(df_tested_r3))
df_tested_r3["fitness"] = (0.25 * df_tested_r3["predicted_fitness"].values + 0.7 + noise).clip(min=0.01)
df_tested_r3["round"] = 3
df_tested_r3["order"] = "triple"

all_tested = pd.concat([all_tested, df_tested_r3[["mutations", "fitness", "round", "order"]]], ignore_index=True)

print(f"\n✅ Round 3: Tested {N_R3} triple mutants")
print(f"   Best fitness: {df_tested_r3['fitness'].max():.3f}")
print(f"   Top 5 triples:")
df_tested_r3.nlargest(5, "fitness")[["mutations", "predicted_fitness", "fitness"]]

---
<div style="background: linear-gradient(135deg, #0A1628, #0F2033); padding: 20px 25px; border-radius: 10px; border-left: 6px solid #06D6A0; margin: 20px 0;">
<h2 style="color: white; margin: 0;">🔄 ROUND 4 — DESIGN & TEST (Quadruple Mutants)</h2>
<p style="color: #06D6A0; margin: 5px 0 0 0;">Final round: retrain on Rounds 1+2+3 → generate quadruples → predict → test</p>
</div>

The final round adds a fourth mutation to the best triples. In the paper, the best AtHMT variant from Round 4 was **V140T/L101I/E206D/S63N** — a quadruple mutant with **16× improvement** in ethyltransferase activity.


In [ ]:
#@title 🔄 Round 4: Train → Generate quadruples → Predict → Test

# ── LEARN ──
model_r4 = train_ridge(all_tested, WT_SEQ)
print(f"✅ Ridge retrained on {len(all_tested)} variants (Rounds 1+2+3)")

# ── DESIGN ──
TOP_TRIPLE_BASES = 6  #@param {type:"integer"}
best_triples = df_tested_r3.nlargest(TOP_TRIPLE_BASES, "fitness")["mutations"].tolist()

quad_candidates = generate_combinatorial_candidates(best_triples, addition_pool, max_candidates=500)
print(f"🔬 Generated {len(quad_candidates)} quadruple mutant candidates")

# ── PREDICT ──
pred_fitness_r4 = predict_fitness(model_r4, quad_candidates, WT_SEQ)
df_quads = pd.DataFrame({"mutations": quad_candidates, "predicted_fitness": pred_fitness_r4})
df_quads = df_quads.sort_values("predicted_fitness", ascending=False).reset_index(drop=True)

# ── TEST ──
N_R4 = 90  #@param {type:"integer"}
df_tested_r4 = df_quads.head(N_R4).copy()
noise = np.random.normal(0, 0.15, size=len(df_tested_r4))
df_tested_r4["fitness"] = (0.2 * df_tested_r4["predicted_fitness"].values + 0.8 + noise).clip(min=0.01)
df_tested_r4["round"] = 4
df_tested_r4["order"] = "quadruple"

all_tested = pd.concat([all_tested, df_tested_r4[["mutations", "fitness", "round", "order"]]], ignore_index=True)

print(f"\n✅ Round 4: Tested {N_R4} quadruple mutants")
print(f"   Best fitness: {df_tested_r4['fitness'].max():.3f}")
print(f"   🏆 Best quadruple mutant:")
best = df_tested_r4.nlargest(1, "fitness").iloc[0]
print(f"   {best['mutations']} → fitness = {best['fitness']:.3f}")

---
<div style="background: linear-gradient(135deg, #0A1628, #0F2033); padding: 20px 25px; border-radius: 10px; border-left: 6px solid #0891B2; margin: 20px 0;">
<h2 style="color: white; margin: 0;">📈 EVALUATION — Fitness Improvement Across All 4 Rounds</h2>
<p style="color: #0891B2; margin: 5px 0 0 0;">Violin plot of fitness by round — same as Figure 4c in Singh et al. (2025)</p>
</div>

This is the key result plot from the paper: a violin plot showing that **fitness improves with each round** as the model accumulates data and explores higher-order mutants.


In [ ]:
#@title 📊 Fitness improvement across 4 rounds (violin plot)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# ── Panel A: Violin plot by round (mirrors Fig 4c from the paper) ──
ax = axes[0]
round_data = [all_tested[all_tested["round"]==r]["fitness"].values for r in [1,2,3,4]]
parts = ax.violinplot(round_data, positions=[1,2,3,4], showmeans=True, showmedians=True)

colors = ["#0891B2", "#06D6A0", "#F4A261", "#E879F9"]
for i, pc in enumerate(parts["bodies"]):
    pc.set_facecolor(colors[i])
    pc.set_alpha(0.7)

ax.set_xticks([1,2,3,4])
ax.set_xticklabels(["Round 1\n(singles)", "Round 2\n(doubles)", "Round 3\n(triples)", "Round 4\n(quads)"])
ax.set_ylabel("Fitness (simulated)", fontsize=12)
ax.set_title("Fitness Distribution by Round", fontweight="bold", fontsize=13)
ax.axhline(y=all_tested[all_tested["round"]==1]["fitness"].median(), color="gray", ls="--", alpha=0.4, label="R1 median")
ax.legend()

# ── Panel B: Best fitness per round ──
ax2 = axes[1]
best_per_round = [all_tested[all_tested["round"]==r]["fitness"].max() for r in [1,2,3,4]]
bars = ax2.bar([1,2,3,4], best_per_round, color=colors, alpha=0.8, edgecolor="white", linewidth=1.5)
ax2.set_xticks([1,2,3,4])
ax2.set_xticklabels(["R1\nSingle", "R2\nDouble", "R3\nTriple", "R4\nQuadruple"])
ax2.set_ylabel("Best Fitness in Round", fontsize=12)
ax2.set_title("Peak Fitness by Round", fontweight="bold", fontsize=13)
for i, v in enumerate(best_per_round):
    ax2.text(i+1, v + 0.02, f"{v:.2f}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout(); plt.show()

# ── Summary table ──
print(f"\n{'='*65}")
print(f"  📊 MULTI-ROUND DBTL SUMMARY")
print(f"{'='*65}")
for r in [1,2,3,4]:
    rd = all_tested[all_tested["round"]==r]
    order = rd["order"].iloc[0]
    best_mut = rd.nlargest(1, "fitness").iloc[0]
    print(f"  Round {r} ({order:10s}): tested={len(rd):3d} | best={best_mut['fitness']:.3f} | {best_mut['mutations']}")
print(f"{'='*65}")
print(f"  Total variants screened: {len(all_tested)}")
print(f"  Cumulative data used for final model: {len(all_tested)} data points")

---
<div style="background: white; padding: 20px 25px; border-radius: 10px; border: 2px solid #E8EFF5;">
<h3 style="color: #0A1628; border-bottom: 3px solid #06D6A0; padding-bottom: 8px;">📈 Accuracy vs. Enrichment — Why Low Correlation is OK</h3>

The paper explicitly notes: *"Although there was little overall consistency between observed enzyme activities and the predictions of the low-N model, the model was nonetheless able to predict a subset of mutants with substantially improved activity in each round."*

This is the **enrichment** concept — the model doesn't need high correlation, it just needs to surface good variants near the top.
</div>

In [ ]:
#@title 📊 Enrichment analysis: ML top picks vs random (Round 4)
fig, ax = plt.subplots(figsize=(6, 5))

top_k = min(15, len(df_tested_r4))
top_pred = df_tested_r4.nlargest(top_k, "predicted_fitness")
rand = df_tested_r4.sample(top_k, random_state=42)

bp = ax.boxplot([top_pred["fitness"], rand["fitness"]],
                labels=["Top ML\npredictions", "Random\nselection"],
                patch_artist=True, widths=0.5)
bp["boxes"][0].set(facecolor="#06D6A0", alpha=0.8)
bp["boxes"][1].set(facecolor="#F4A261", alpha=0.8)
ax.set_ylabel("Measured Fitness (simulated)")
ax.set_title("Round 4: ML Top-K vs Random", fontweight="bold")
plt.tight_layout(); plt.show()

enrich = top_pred["fitness"].mean() / max(rand["fitness"].mean(), 1e-6)
print(f"Top-ML mean: {top_pred['fitness'].mean():.3f} | Random mean: {rand['fitness'].mean():.3f} | Enrichment: {enrich:.2f}×")

---
<div style="background: linear-gradient(135deg, #0A1628, #0F2033); padding: 25px 30px; border-radius: 12px;">
<h2 style="color: white; margin: 0 0 12px 0;">🎓 Summary — The Complete 4-Round DBTL Cycle</h2>

<table style="width:100%; border-collapse: separate; border-spacing: 8px;">
<tr>
<td style="background: rgba(8,145,178,0.15); padding: 14px; border-radius: 8px; border-left: 4px solid #0891B2; vertical-align:top;">
<strong style="color: #0891B2;">Round 1: Singles</strong><br>
<span style="color: #CBD5E1; font-size: 13px;">ESM-2 zero-shot scoring → 180 single mutants tested → best: V140T (2.1×)</span>
</td>
<td style="background: rgba(6,214,160,0.12); padding: 14px; border-radius: 8px; border-left: 4px solid #06D6A0; vertical-align:top;">
<strong style="color: #06D6A0;">Round 2: Doubles</strong><br>
<span style="color: #CBD5E1; font-size: 13px;">Ridge trained on R1 → best singles + new mutation → 90 doubles → best: S99T/V140T (3×)</span>
</td>
</tr>
<tr>
<td style="background: rgba(244,162,97,0.12); padding: 14px; border-radius: 8px; border-left: 4px solid #F4A261; vertical-align:top;">
<strong style="color: #F4A261;">Round 3: Triples</strong><br>
<span style="color: #CBD5E1; font-size: 13px;">Ridge retrained on R1+R2 → best doubles + new mutation → 90 triples → 82% beat V140T alone</span>
</td>
<td style="background: rgba(232,121,249,0.12); padding: 14px; border-radius: 8px; border-left: 4px solid #E879F9; vertical-align:top;">
<strong style="color: #E879F9;">Round 4: Quadruples</strong><br>
<span style="color: #CBD5E1; font-size: 13px;">Ridge retrained on R1+R2+R3 → best triples + new mutation → V140T/L101I/E206D/S63N (16×)</span>
</td>
</tr>
</table>

<div style="margin-top: 15px; padding: 12px 16px; background: rgba(255,255,255,0.05); border-radius: 6px;">
<span style="color: #94A3B8; font-size: 13px;">
<strong style="color: white;">Key takeaways:</strong><br>
• <strong>Each round adds one mutation</strong> — single → double → triple → quadruple (as in Singh et al. 2025)<br>
• <strong>Ridge is retrained cumulatively</strong> — using ALL data from previous rounds, not just the latest<br>
• <strong>The model gets smarter</strong> — it learns which mutation combinations are synergistic<br>
• <strong>Enrichment matters more than accuracy</strong> — even weak correlation can identify improved variants<br>
• <strong>4 weeks, <500 variants</strong> — achieved 16× (AtHMT) and 26× (YmPhytase) improvements
</span>
</div>

<p style="color: #64748B; margin: 12px 0 0 0; font-size: 12px;">
<a href="https://doi.org/10.1038/s41467-025-61209-y" style="color: #06D6A0;"></a>
<a href="https://github.com/Zhao-Group/closed-loop" style="color: #06D6A0;"></a>
<a href="https://doi.org/10.1038/s41587-021-01146-5" style="color: #06D6A0;"></a>
<a href="https://github.com/facebookresearch/esm" style="color: #06D6A0;"></a>
</p>
</div>